In [1]:
import torch
import torch.nn as nn

In [23]:
class MultiHeadAttention(nn.Module):
  def __init__(self, din, dout, context_length, num_heads, dropout, qkv_bias=False):
    super().__init__()
    self.dout = dout
    self.head_dim = dout // num_heads
    self.num_heads = num_heads
    self.wk = nn.Linear(din, dout, bias=qkv_bias)
    self.wq = nn.Linear(din, dout, bias=qkv_bias)
    self.wv = nn.Linear(din, dout, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.mask = torch.triu(torch.ones(context_length, context_length), diagonal = 1)

  def forward(self, x):
    b, num_tokens, din = x.shape
    keys = self.wk(x) # b, num_tokens, dout      dout = num_heads * head_dim
    queries = self.wq(x)
    values = self.wv(x)

    #Changing their dim
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Grouping  by head
    keys = keys.transpose(1,2) # b, num_heads, num_tokens, head_dim (1,2,3,3)
    queries = queries.transpose(1,2)
    values = values.transpose(1, 2)

    # Finding attention scores
    attention_scores = queries @ keys.transpose(2,3) # 1, 2 , 3, 3 (b, num_heads, num_tokens, num_tokens)
    attention_scores.masked_fill_(self.mask.bool(), -torch.inf)
    # print(attention_scores)
    attention_weights = torch.softmax(attention_scores / (keys.shape[-1] ** 0.5), dim = -1)
    attention_weights = self.dropout(attention_weights)

    # Computing context vector
    context_vector = attention_weights @ values
    context_vector = context_vector.transpose(1,2).contiguous().view(b, num_tokens, self.dout)
    return context_vector

In [8]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],  # Row 1
     [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],  # Row 2
     [0.77, 0.25, 0.10, 0.05, 0.80, 0.55]]  # Row 3
)

batch = torch.stack((inputs, inputs), dim = 0)
print(batch)

tensor([[[0.4300, 0.1500, 0.8900, 0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400, 0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000, 0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900, 0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400, 0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000, 0.0500, 0.8000, 0.5500]]])


In [31]:
batch_size, context_length, din = batch.shape
dout = 6
num_heads = 2
mha = MultiHeadAttention(din, dout, context_length, num_heads, 0.0)
context_vectors = mha.forward(batch)
print(context_vectors)

tensor([[[ 0.2638,  0.3909, -0.0105, -0.1606, -0.2744, -0.8912],
         [ 0.2559,  0.3541,  0.0841, -0.2548, -0.3622, -0.6477],
         [ 0.3735,  0.2723, -0.0420, -0.2521, -0.3666, -0.5682]],

        [[ 0.2638,  0.3909, -0.0105, -0.1606, -0.2744, -0.8912],
         [ 0.2559,  0.3541,  0.0841, -0.2548, -0.3622, -0.6477],
         [ 0.3735,  0.2723, -0.0420, -0.2521, -0.3666, -0.5682]]],
       grad_fn=<ViewBackward0>)
